In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter("ignore")
from sklearn.preprocessing  import LabelEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn import tree 
from sklearn import ensemble 
from sklearn import metrics 
from sklearn.model_selection import train_test_split 
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV

In [18]:
df = pd.read_csv('../data/bank_preprocessed.csv')
df.head()

,age,balance,day,duration,campaign,pdays,previous,deposit,job_admin.,job_blue-collar,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
0,59,2343.0,5,1042,1,-1,0,1,True,False,...,False,False,True,False,False,False,False,False,False,True
1,56,45.0,5,1467,1,-1,0,1,True,False,...,False,False,True,False,False,False,False,False,False,True
2,41,1270.0,5,1389,1,-1,0,1,False,False,...,False,False,True,False,False,False,False,False,False,True
3,55,2476.0,5,579,1,-1,0,1,False,False,...,False,False,True,False,False,False,False,False,False,True
4,54,184.0,5,673,2,-1,0,1,True,False,...,False,False,True,False,False,False,False,False,False,True


In [ ]:
#Отделим целевой признак
X = df.drop(['deposit'], axis=1)
y = df['deposit']

#Сделаем стратифицированное разбиение на тренировочную и тестовую выборки 67\33
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42,
    stratify=y
)

In [ ]:
#Отберем 15 самых важных признаков
#Инициализируем селектор
selector = SelectKBest(score_func=f_classif, k=15)

#Обучим селектор
selector.fit(X_train, y_train)

#Применяем селектор к выборкам
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

In [ ]:
#Стандартизируем данные
#Инициализируем скейлер
scaler = MinMaxScaler()
#Применяем к выборкам
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

In [ ]:
#Обучим первую модель логистическую регрессию
#Инициализируем модель
lr = LogisticRegression(
    solver='sag',
    random_state=42,
    max_iter=1000
)

#Обучаем модель
lr.fit(X_train_scaled, y_train)
#Делаем предсказания
y_pred = lr.predict(X_test_scaled)
#Смотрим метрику accuracy
accuracy = metrics.accuracy_score(y_test, y_pred)

round(accuracy, 3)

0.802

На тестовой выборке значение метрики 0.802, что очень даже неплохо, особенно учитывая, что это без подбора гиперпараметров, посмотрим как дерево решений справится с этой же задачей

In [ ]:
#Инициализируем дерево
model_tree = tree.DecisionTreeClassifier(
    criterion='entropy',
    random_state=42,
    max_depth=6
)

#Обучаем дерево
model_tree.fit(X_train_scaled, y_train)
#Делаем предсказание
y_pred_tree = model_tree.predict(X_test_scaled)
#Смотрим метрику
accuracy = metrics.accuracy_score(y_test, y_pred_tree)
round(accuracy, 3)

0.804

Совсем немного, но лучше, чем регрессия, теперь попробуем подобрать гиперпараметры

In [ ]:
#Задаем сетку параметров
param_grid = {'min_samples_split': [2, 5, 7, 10],
              'max_depth': [3,5,7]
              }

#Инициализируем грид сирч
grid_search = GridSearchCV(
    estimator=tree.DecisionTreeClassifier(
        random_state=42,
        criterion='entropy'
    ),
    param_grid=param_grid,
    n_jobs=-1
)

#Обучаем и смотрим на метрики
%time grid_search.fit(X_train_scaled, y_train) 
print("accuracy на тестовом наборе: {:.2f}".format(grid_search.score(X_test_scaled, y_test)))
y_test_pred = grid_search.predict(X_test_scaled)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(grid_search.best_params_))

CPU times: total: 78.1 ms
Wall time: 104 ms
accuracy на тестовом наборе: 0.80
f1_score на тестовом наборе: 0.78
Наилучшие значения гиперпараметров: {'max_depth': 7, 'min_samples_split': 2}


Из заданной сетки грид сирч нашел оптимальные гиперпараметры, но они чуть хуже стандартных, далее попробуем уже более продвинутые методы

In [38]:
pd.DataFrame(X_train_scaled).to_csv('../data/X_train_scaled.csv', index=False)
pd.DataFrame(X_test_scaled).to_csv('../data/X_test_scaled.csv', index=False)

y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)